# Generate Two-Circle Point Cloud Test Data

This notebook creates a synthetic point cloud on two slightly separated circles and saves it as a plain text file with columns:

```text
x y f
```

The scalar function value `f` is computed from the local geometry: for each point, `f` is the sum of the Euclidean distances to its nearest `q` neighbors.

## Configuration

Adjust the parameters below to control the number of points, circle sizes, noise level, KNN parameter `q`, and output path.

In [ ]:
from pathlib import Path
import os

import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "data" / "pointcloud_examples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MATPLOTLIB_CACHE_DIR = PROJECT_ROOT / "outputs" / "matplotlib_cache"
MATPLOTLIB_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MATPLOTLIB_CACHE_DIR))


OUTPUT_PATH = OUTPUT_DIR / "two_circles_200pts_knn_function.txt"

RANDOM_SEED = 20260611

TOTAL_POINTS = 200
BIG_CIRCLE_POINTS = 140
SMALL_CIRCLE_POINTS = TOTAL_POINTS - BIG_CIRCLE_POINTS

BIG_CENTER = (0.0, 0.0)
BIG_RADIUS = 1.0
BIG_RADIAL_NOISE = 0.035
BIG_XY_NOISE = 0.015

SMALL_CENTER = (1.55, 0.15)
SMALL_RADIUS = 0.35
SMALL_RADIAL_NOISE = 0.018
SMALL_XY_NOISE = 0.01

# Number of nearest neighbors used in the function value.
KNN_Q = 8

# If true, rescale function values to [0, 1] after KNN computation.
NORMALIZE_FUNCTION_VALUES = False

print("Output path:", OUTPUT_PATH)

## Sample the Two Circles

The large circle and the small circle are sampled independently. Small radial and coordinate noise are added so that the data are not perfectly regular.

In [ ]:
def sample_noisy_circle(
    rng: np.random.Generator,
    n_points: int,
    *,
    center: tuple[float, float],
    radius: float,
    radial_noise: float,
    xy_noise: float,
) -> np.ndarray:
    angles = rng.uniform(0.0, 2.0 * np.pi, size=n_points)
    noisy_radii = radius + rng.normal(0.0, radial_noise, size=n_points)
    circle = np.column_stack((np.cos(angles), np.sin(angles))) * noisy_radii[:, None]
    coordinates = circle + np.asarray(center, dtype=float)
    coordinates += rng.normal(0.0, xy_noise, size=(n_points, 2))
    return coordinates


rng = np.random.default_rng(RANDOM_SEED)

big_circle = sample_noisy_circle(
    rng,
    BIG_CIRCLE_POINTS,
    center=BIG_CENTER,
    radius=BIG_RADIUS,
    radial_noise=BIG_RADIAL_NOISE,
    xy_noise=BIG_XY_NOISE,
)
small_circle = sample_noisy_circle(
    rng,
    SMALL_CIRCLE_POINTS,
    center=SMALL_CENTER,
    radius=SMALL_RADIUS,
    radial_noise=SMALL_RADIAL_NOISE,
    xy_noise=SMALL_XY_NOISE,
)

points = np.vstack((big_circle, small_circle))
labels = np.array([0] * BIG_CIRCLE_POINTS + [1] * SMALL_CIRCLE_POINTS)

permutation = rng.permutation(len(points))
points = points[permutation]
labels = labels[permutation]

print("points shape:", points.shape)
print("big circle points:", int(np.sum(labels == 0)))
print("small circle points:", int(np.sum(labels == 1)))

## Compute KNN Distance-Sum Function Values

For each point `p_i`, sort all distances from `p_i` to the other points and sum the first `q` distances:

```text
f_i = sum_{p_j in q nearest neighbors of p_i} ||p_i - p_j||_2
```

This is a simple local sparsity function: points in denser regions tend to have smaller values.

In [ ]:
def knn_distance_sum(points: np.ndarray, q: int) -> np.ndarray:
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("points must have shape (n_points, 2).")
    if not 1 <= q < len(points):
        raise ValueError("q must satisfy 1 <= q < number of points.")

    differences = points[:, None, :] - points[None, :, :]
    distances = np.linalg.norm(differences, axis=2)
    sorted_distances = np.sort(distances, axis=1)
    nearest_distances = sorted_distances[:, 1 : q + 1]
    return nearest_distances.sum(axis=1)


function_values = knn_distance_sum(points, KNN_Q)

if NORMALIZE_FUNCTION_VALUES:
    min_value = float(function_values.min())
    max_value = float(function_values.max())
    if max_value > min_value:
        function_values = (function_values - min_value) / (max_value - min_value)

print("q:", KNN_Q)
print("function min:", float(function_values.min()))
print("function max:", float(function_values.max()))
print("function mean:", float(function_values.mean()))

## Save the Point Cloud

The saved file is compatible with `pointcloud_to_scc2020.py` using the default `FUNCTION_MODE = "last"`.

In [ ]:
data = np.column_stack((points, function_values))

header_lines = [
    "# Two-circle point cloud with KNN distance-sum function values",
    "# columns: x y f",
    f"# total_points: {len(points)}",
    f"# big_circle_points: {BIG_CIRCLE_POINTS}",
    f"# small_circle_points: {SMALL_CIRCLE_POINTS}",
    f"# big_center: {BIG_CENTER}",
    f"# big_radius: {BIG_RADIUS}",
    f"# small_center: {SMALL_CENTER}",
    f"# small_radius: {SMALL_RADIUS}",
    f"# knn_q: {KNN_Q}",
    f"# normalized_function_values: {NORMALIZE_FUNCTION_VALUES}",
]

rows = [f"{x:.17g} {y:.17g} {f_value:.17g}" for x, y, f_value in data]
OUTPUT_PATH.write_text("\n".join(header_lines + rows) + "\n", encoding="utf-8")

print("Saved:", OUTPUT_PATH)
print("Rows:", len(rows))

## Preview the Saved File

In [ ]:
saved_lines = OUTPUT_PATH.read_text(encoding="utf-8").splitlines()
print("\n".join(saved_lines[:20]))

## Visualize the Point Cloud

The plot colors each point by its KNN distance-sum function value.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(points[:, 0], points[:, 1], c=function_values, s=22, cmap="viridis")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Two circles, q={KNN_Q}")
fig.colorbar(scatter, ax=ax, label="KNN distance-sum function value")
plt.show()

## Use This File in the scc2020 Converter Notebook

In `Pointcloud_to_scc2020_tutorial.ipynb`, set:

```python
INPUT_POINTCLOUD = PROJECT_ROOT / "data" / "pointcloud_examples" / "two_circles_200pts_knn_function.txt"
FUNCTION_MODE = "last"
FILTRATION_METHOD = "alpha"  # or "rips"
```

Then run the conversion cells to save the corresponding `scc2020` chain complex.